In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split


In [ ]:
reviews = pd.read_csv("../data/processed_review_data.csv")
restaurants = pd.read_csv("../data/processed_restaurant_data.csv")
restaurants_tp = pd.read_csv("../data/yelp_restaurant_data.csv")
restaurants = restaurants.merge(
                    restaurants_tp[[
                    "id", "transactions", "price"
                    ]], on="id", how="left")
restaurants.rename(columns={"id": "restaurant_id"}, inplace=True)
TRANSACTION_TYPES = ['pickup', 'delivery', 'restaurant_reservation']
for t in TRANSACTION_TYPES:
    restaurants[f'is_{t}'] = restaurants['transactions'].apply(lambda lst: 1 if t in lst else 0)
PRICE_MAP = {'$': 1, '$$': 2, '$$$': 3, '$$$$':4}
restaurants['price_cat'] = restaurants['price'].map(PRICE_MAP).fillna(0).astype(int)

In [633]:
restaurants.columns

Index(['Unnamed: 0', 'restaurant_id', 'name', 'categories', 'rating',
       'review_count', 'latitude', 'longitude', 'categories_list',
       'log_review_count', 'normalized_rating', 'normalized_log_review_count',
       'popularity_score', 'wilson_score', 'normalized_wilson_score',
       'normalized_latitude', 'normalized_longitude', 'transactions', 'price',
       'is_pickup', 'is_delivery', 'is_restaurant_reservation', 'price_cat'],
      dtype='object')

In [634]:
restaurants.head(2)

,Unnamed: 0,restaurant_id,name,categories,rating,review_count,latitude,longitude,categories_list,log_review_count,...,wilson_score,normalized_wilson_score,normalized_latitude,normalized_longitude,transactions,price,is_pickup,is_delivery,is_restaurant_reservation,price_cat
0,0,7E0GO6fb7KIGAd9js7mDPg,Jamaica Jerk Villa,"[{'alias': 'caribbean', 'title': 'Caribbean'}]",3.1,163,41.75042,-87.64308,['Caribbean'],5.099866,...,3.061677,0.397850,-0.013719,-0.177156,"['pickup', 'delivery']",$$,1,1,0,2
1,1,NYY3GwZTg332dkhs8JJnlg,Italian Fiesta Pizzeria,"[{'alias': 'pizza', 'title': 'Pizza'}, {'alias...",2.2,112,41.74729,-87.64431,"['Pizza', 'Italian']",4.727388,...,2.175862,0.218197,-0.014990,-0.181499,"['pickup', 'delivery']",$$,1,1,0,2


In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

def safe_parse_categories(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else x
    except:
        return []
def extract_titles(category_list):
    if not isinstance(category_list, list):
        return []
    return [d.get("title") for d in category_list if isinstance(d, dict) and "title" in d]

restaurants['parsed_categories'] = restaurants['categories'].apply(safe_parse_categories)
restaurants['category_titles'] = restaurants['parsed_categories'].apply(extract_titles)

all_categories = set(cat for sublist in restaurants['category_titles'] for cat in sublist)
category_vocab = {cat: idx + 1 for idx, cat in enumerate(sorted(all_categories))}  # 保留 0 給 padding
category_vocab_size = len(category_vocab) + 1
MAX_CATEGORIES = 5

def map_to_ids(cat_list):
    return [category_vocab[c] for c in cat_list if c in category_vocab]

from tensorflow.keras.preprocessing.sequence import pad_sequences
restaurants['category_ids'] = restaurants['category_titles'].apply(map_to_ids)
restaurants['category_ids_padded'] = pad_sequences(
    restaurants['category_ids'], maxlen=MAX_CATEGORIES, padding='post'
).tolist()

In [663]:
restaurants.head()

,restaurant_id,latitude,longitude,normalized_rating,normalized_log_review_count,popularity_score,normalized_wilson_score,is_pickup,is_delivery,is_restaurant_reservation,price_cat,parsed_categories,category_titles,category_ids,category_ids_padded
0,7E0GO6fb7KIGAd9js7mDPg,41.750420,-87.643080,0.62,0.553837,0.573686,0.397850,1,1,0,2,"[{'alias': 'caribbean', 'title': 'Caribbean'}]",[Caribbean],[76],"[76, 0, 0, 0, 0]"
1,NYY3GwZTg332dkhs8JJnlg,41.747290,-87.644310,0.44,0.513387,0.491371,0.218197,1,1,0,2,"[{'alias': 'pizza', 'title': 'Pizza'}, {'alias...","[Pizza, Italian]","[291, 206]","[291, 206, 0, 0, 0]"
2,zPZuFYVeu6KfQpupYhP1LQ,41.735966,-87.668341,0.42,0.451648,0.442154,0.195546,0,0,0,0,"[{'alias': 'southern', 'title': 'Southern'}, {...","[Southern, Soul Food]","[343, 340]","[343, 340, 0, 0, 0]"
3,JwdC0viyk-edOO9X-X413w,41.745622,-87.643690,0.70,0.429099,0.510369,0.457247,0,1,0,1,"[{'alias': 'sandwiches', 'title': 'Sandwiches'}]",[Sandwiches],[323],"[323, 0, 0, 0, 0]"
4,q_QEtAn21JUaDi08UQDqTA,41.747325,-87.663593,0.66,0.405905,0.482133,0.413735,1,1,0,0,"[{'alias': 'comfortfood', 'title': 'Comfort Fo...","[Comfort Food, Cajun/Creole, Soul Food]","[97, 68, 340]","[97, 68, 340, 0, 0]"


In [637]:
restaurants = restaurants.drop(columns=['Unnamed: 0','name', 'categories', 'rating', 'review_count', 'categories_list','log_review_count','wilson_score', 'normalized_latitude', 'normalized_longitude', 'transactions', 'price'])

In [638]:
restaurants.columns

Index(['restaurant_id', 'latitude', 'longitude', 'normalized_rating',
       'normalized_log_review_count', 'popularity_score',
       'normalized_wilson_score', 'is_pickup', 'is_delivery',
       'is_restaurant_reservation', 'price_cat', 'parsed_categories',
       'category_titles', 'category_ids', 'category_ids_padded'],
      dtype='object')

In [639]:
df = reviews.merge(
    restaurants,
    on="restaurant_id", how="left"
)

In [640]:
df.columns

Index(['restaurant_id', 'review_id', 'rating', 'text', 'time_created',
       'user_id', 'user_name', 'bert_sentiment', 'bert_score',
       'mapped_sentiment', 'weighted_score', 'days_since_review',
       'recency_score', 'normalized_rating_x', 'latitude', 'longitude',
       'normalized_rating_y', 'normalized_log_review_count',
       'popularity_score', 'normalized_wilson_score', 'is_pickup',
       'is_delivery', 'is_restaurant_reservation', 'price_cat',
       'parsed_categories', 'category_titles', 'category_ids',
       'category_ids_padded'],
      dtype='object')

In [641]:
df[['normalized_rating_x','normalized_rating_y']]
df = df.rename(columns={"normalized_rating_x": "customer_normalized_rating", "normalized_rating_y": "restaurant_normalized_rating"})
sentiment_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['mapped_sentiment_num'] = df['mapped_sentiment'].map(sentiment_map)
df['label_binary'] = (df['customer_normalized_rating'] >= 0.4).astype(int)

In [642]:
EMBED_DIM = 8
DNN_UNITS = [64, 32]
DROPOUT_RATE = 0.3
USER_EMB_DIM = 4
PRICE_EMB_DIM = 4
CATEGORY_EMB_DIM = 8


In [643]:
user_encoder = LabelEncoder()
df['user_id_encoded'] = user_encoder.fit_transform(df['user_id'])
user_vocab_size = df['user_id_encoded'].nunique() + 1
price_vocab_size = 4
# category_cols = list(category_df.columns)
numeric_cols = ['restaurant_normalized_rating','normalized_log_review_count',
       'popularity_score','normalized_wilson_score','is_pickup', 
       'is_delivery', 'is_restaurant_reservation'] 


In [644]:
# input layer
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate, Flatten, Dropout, Reshape
user_id_in = Input(shape=(1,), name='user_id', dtype='int32')
price_in = Input(shape=(1,), name='price_cat', dtype='int32')
category_in = Input(shape=(MAX_CATEGORIES,), name='category_ids', dtype='int32')
numeric_inputs = [Input(shape=(1,), name=col, dtype='float32') for col in numeric_cols]


In [645]:
# embedding # dense
from tensorflow.keras.layers import (
    Input, Embedding, Dense, Concatenate, Flatten, Dropout,
    Reshape, Lambda
)

user_emb = Embedding(user_vocab_size, USER_EMB_DIM)(user_id_in)
price_emb = Embedding(price_vocab_size, PRICE_EMB_DIM)(price_in)

cat_emb = Embedding(category_vocab_size, CATEGORY_EMB_DIM)(category_in)  
cat_emb_pooled = Lambda(
    lambda x: K.mean(x, axis=1, keepdims=True),
    output_shape=(1, EMBED_DIM)
)(cat_emb)
UNIFIED_EMB_DIM = 8
user_emb = Dense(UNIFIED_EMB_DIM)(user_emb)
price_emb = Dense(UNIFIED_EMB_DIM)(price_emb)

num_embs = []
for inp in numeric_inputs:
    x = Reshape((1,1))(inp)
    x = Dense(EMBED_DIM)(x)
    num_embs.append(x)

In [646]:
# FM
all_embs = [user_emb, price_emb, cat_emb_pooled] + num_embs
fm_input = Concatenate(axis=1)(all_embs)

sum_sq = Lambda(
    lambda x: K.square(K.sum(x, axis=1)),
    output_shape=(EMBED_DIM,)
)(fm_input)

sq_sum = Lambda(
    lambda x: K.sum(K.square(x), axis=1),
    output_shape=(EMBED_DIM,)
)(fm_input)
cross = Lambda(
    lambda t: 0.5 * K.sum(t[0] - t[1], axis=1, keepdims=True),
    output_shape=(1,)
)([sum_sq, sq_sum])

In [647]:
# flattening for dense 96->64->32
deep_x = Flatten()(fm_input)
for n in DNN_UNITS:
    deep_x = Dense(n, activation='relu')(deep_x)
    deep_x = Dropout(DROPOUT_RATE)(deep_x)
deep_out = Dense(1)(deep_x)

In [648]:
# concat FM and deep
logits = Concatenate(axis=1)([cross, deep_out])
output = Dense(1, activation='sigmoid')(logits)

In [649]:
from tensorflow.keras.models import Model
model = Model(inputs=[user_id_in, price_in, category_in] + numeric_inputs, outputs=output)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC', 'accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_id             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price_cat           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ category_ids        │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ restaurant_normali… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalized_log_rev… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ popularity_score    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalized_wilson_… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ is_pickup           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ is_delivery         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ is_restaurant_rese… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 4)      │     37,976 │ user_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 4)      │         16 │ price_cat[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 5, 8)      │      3,344 │ category_ids[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1)      │          0 │ restaurant_norma… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 1)      │          0 │ normalized_log_r… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 1, 1)      │          0 │ popularity_score… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 1, 1)      │          0 │ normalized_wilso… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_4 (Reshape) │ (None, 1, 1)      │          0 │ is_pickup[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 48,828 (190.73 KB)

 Trainable params: 48,828 (190.73 KB)

 Non-trainable params: 0 (0.00 B)

In [650]:
from sklearn.model_selection import train_test_split

y = df['label_binary']
X = {
    'user_id': df['user_id_encoded'].astype(int),
    'price_cat': df['price_cat'].astype(int),
    'category_ids': np.stack(df['category_ids_padded'].values)
}

for col in numeric_cols:
    X[col] = df[col].astype(float)

In [651]:
X_train, X_val, y_train, y_val = {}, {}, None, None
for key in X:
    X_train[key], X_val[key], y_train, y_val = train_test_split(
        X[key], y, test_size=0.2, random_state=42
    )

In [652]:
from tensorflow.keras import backend as K
K.clear_session()
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=256,
    epochs=5
)

Epoch 1/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - AUC: 0.5875 - accuracy: 0.7549 - loss: 0.5388 - val_AUC: 0.8000 - val_accuracy: 0.7836 - val_loss: 0.4486
Epoch 2/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.7747 - accuracy: 0.7914 - loss: 0.4443 - val_AUC: 0.8149 - val_accuracy: 0.8016 - val_loss: 0.4203
Epoch 3/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.8246 - accuracy: 0.8185 - loss: 0.3983 - val_AUC: 0.8293 - val_accuracy: 0.8367 - val_loss: 0.3958
Epoch 4/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.9107 - accuracy: 0.8693 - loss: 0.3142 - val_AUC: 0.8074 - val_accuracy: 0.8227 - val_loss: 0.4187
Epoch 5/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.9741 - accuracy: 0.9342 - loss: 0.1764 - val_AUC: 0.7710 - val_accuracy: 0.7994 - val_loss: 0.5345


In [653]:
import numpy as np
import pandas as pd

def recommend_topk(
    model,
    user_encoder,
    df_restaurants,
    user_id,
    pref_categories=None,
    price_cat=None,
    is_delivery=None,
    is_pickup=None,
    min_pred_rating=0.6,
    top_k=10,
    category_vocab=None,
    MAX_CATEGORIES=5
):
    df = df_restaurants.copy()

    if user_id in user_encoder.classes_:
        user_id_encoded = user_encoder.transform([user_id])[0]
    else:
        user_id_encoded = 0

    df['user_id_encoded'] = user_id_encoded

    if pref_categories:
        df = df[df['category_titles'].apply(lambda x: any(cat in x for cat in pref_categories))]

    if price_cat is not None:
        df = df[df['price_cat'] == price_cat]

    if is_delivery is not None:
        df = df[df['is_delivery'] == int(is_delivery)]
    if is_pickup is not None:
        df = df[df['is_pickup'] == int(is_pickup)]

    if df.empty:
        return pd.DataFrame()

    def map_cats_to_ids(cats):
        return [category_vocab[c] for c in cats if c in category_vocab]

    df['category_ids'] = df['category_titles'].apply(map_cats_to_ids)
    df['category_ids_padded'] = df['category_ids'].apply(
        lambda x: x[:MAX_CATEGORIES] + [0] * (MAX_CATEGORIES - len(x)) if len(x) < MAX_CATEGORIES else x[:MAX_CATEGORIES]
    )
    cat_array = np.stack(df['category_ids_padded'].values)

    model_input = {
        'user_id': np.full((len(df),), user_id_encoded),
        'price_cat': df['price_cat'].values,
        'category_ids': cat_array
    }

    numeric_cols = [
        'restaurant_normalized_rating', 'normalized_log_review_count', 'popularity_score',
        'normalized_wilson_score', 'is_pickup', 'is_delivery', 'is_restaurant_reservation'
    ]
    for col in numeric_cols:
        model_input[col] = df[col].values

    preds = model.predict(model_input, verbose=0).reshape(-1)
    df['predicted_rating'] = preds

    df = df[df['predicted_rating'] >= min_pred_rating]
    df = df.sort_values(by='predicted_rating', ascending=False)
    df = df.drop_duplicates(subset='restaurant_id', keep='first')
    df = df.head(top_k)

    return df[['restaurant_id', 'category_titles', 'price_cat' ,'predicted_rating', 'label_binary']]

In [654]:
df.columns

Index(['restaurant_id', 'review_id', 'rating', 'text', 'time_created',
       'user_id', 'user_name', 'bert_sentiment', 'bert_score',
       'mapped_sentiment', 'weighted_score', 'days_since_review',
       'recency_score', 'customer_normalized_rating', 'latitude', 'longitude',
       'restaurant_normalized_rating', 'normalized_log_review_count',
       'popularity_score', 'normalized_wilson_score', 'is_pickup',
       'is_delivery', 'is_restaurant_reservation', 'price_cat',
       'parsed_categories', 'category_titles', 'category_ids',
       'category_ids_padded', 'mapped_sentiment_num', 'label_binary',
       'user_id_encoded'],
      dtype='object')

In [655]:
top10 = recommend_topk(
    model=model,
    user_encoder=user_encoder,
    df_restaurants=df,
    user_id='tgeFUChlh7v8bZFVl2-hjQ',
    # user_id='nDrqm6a3NXmv7E7ofUkX9w',
    # user_id='s-yGhMIJTcW39FTTzc2_Eg',
    # pref_categories=['New American', 'Bars', 'Cocktail Bars', 'Sushi Bars', 'Sports Bars'],
    pref_categories=['Bars'],
    # pref_categories=['Italian'],
    price_cat=None,
    is_delivery=None,
    is_pickup=None,
    min_pred_rating=0,
    top_k=10,
    category_vocab=category_vocab,
    MAX_CATEGORIES=5
)
top10

,restaurant_id,category_titles,price_cat,predicted_rating,label_binary
4957,apyIymDFbUK7u-Y3V5kmkA,"[Steakhouses, Seafood, Bars]",0,0.999996,1
9455,RnR_zPbkVcG9G52hgYov_g,"[Indian, Bars]",2,0.999994,1
5151,Vw-Qn1Hg06h4yBUdDQCXyA,"[Sushi Bars, Japanese, Bars]",2,0.999994,1
5141,jxPXmzh6a_JZbLDb1QM27w,"[New American, Bars]",2,0.999994,1
5034,JMWtBuuNmTXqoBs06Wt_7w,"[Steakhouses, Bars, Desserts]",0,0.999994,1
5061,1Q7RL7Hc63wKa2A9ec3zXw,"[Italian, Bars, Salad]",2,0.999994,1
817,wfxgPtSkiAfkKNBVEWts9g,"[Bars, Italian]",2,0.999993,1
5174,f1neY8Zd9xld7BB7G7mC0Q,"[Steakhouses, Seafood, Bars]",0,0.999993,1
8858,KXiClWndL6UyPJDt7y50zQ,"[Italian, Bars]",2,0.999993,1
1053,TvOgiK8LIPKWx8hffKSEGw,"[Bars, American]",2,0.999993,0


In [656]:
def precision_at_k(y_true, y_pred, k):
    top_k = np.argsort(y_pred)[-k:][::-1]
    return np.sum(y_true[top_k]) / k

def recall_at_k(y_true, y_pred, k):
    top_k = np.argsort(y_pred)[-k:][::-1]
    relevant = np.sum(y_true)
    return np.sum(y_true[top_k]) / relevant if relevant > 0 else 0

In [ ]:
y_true_10 = top10['label_binary'].values
y_pred_10 = top10['predicted_rating'].values

print("Precision@K:", precision_at_k(y_true_10, y_pred_10, k=10))
print("Recall@K:", recall_at_k(y_true_10, y_pred_10, k=10))

Precision@K: 0.9
Recall@K:    1.0
